# MorphoNup — train and evaluate one dataset

Set the dataset, number of outer folds and seeds below, then run all cells.

* `RUN_MODE = "background"` (recommended): starts `scripts/run_cv.py` detached (`setsid nohup`). It keeps running if the notebook or the session closes. Re-run the *Progress* and *Results* cells any time.
* `RUN_MODE = "inline"`: trains inside this kernel (the kernel must stay alive for the whole run).
* `RUN_MODE = "results_only"`: no training; shows what is saved.

Every run writes its figures automatically (per fold, per seed, over seeds); the Results cells only display them and redraw any that are missing from the saved files. Finished folds are skipped, so relaunching resumes a stopped job.

In [ ]:
DATASET = "DatasetNup_1/Homo_sapiens"   # one of the 12 IDs printed by the next cell
K_FOLDS = 5                              # 5 or 10
SEEDS = [151000, 152000, 153000]          # the reported seeds (config.SEEDS)
RUN_MODE = "results_only"                # "background" | "inline" | "results_only"
ALLOW_BUSY_GPU = False                   # shared account: True only after checking with labmates
SHOW_SEED = SEEDS[0]                     # seed whose per-seed figures are shown below

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
_null, _err = os.open(os.devnull, os.O_WRONLY), os.dup(2)
os.dup2(_null, 2)                                    # TensorFlow's CUDA start-up lines go to C-level stderr
try:
    import tensorflow  # noqa: F401
finally:
    os.dup2(_err, 2)
    os.close(_null)
    os.close(_err)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "morphonup" / "config.py").exists())
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, Markdown, display

from morphonup import config, figures, jobs
from morphonup.cv import run_dir

pd.set_option("display.width", 200, "display.max_columns", 40)


def show(path, width=900):
    """Display a saved figure with its caption (from the folder's figure_catalog.csv)."""
    path = Path(path)
    if not path.exists():
        print("missing:", path.relative_to(ROOT))
        return
    cat = path.parent / "figure_catalog.csv"
    caption = pd.read_csv(cat).set_index("png").suggested_caption.get(path.name, "") if cat.exists() else ""
    display(Markdown(f"**{path.stem}** — `{path.relative_to(ROOT)}`  \n{caption}"))
    display(Image(filename=str(path), width=width))

assert DATASET in config.DATASETS, "\n".join(config.DATASETS)
assert RUN_MODE in ("background", "inline", "results_only"), RUN_MODE
RUN = run_dir(DATASET, K_FOLDS)
fasta = next((config.dataset_folder(DATASET) / "sequence").glob("*.fas"))
headers = [l[1:].strip() for l in fasta.open() if l.startswith(">")]
n, pos = len(headers), sum(h.startswith("nucleosomal_sequence") for h in headers)
print(f"{DATASET}: {n:,} sequences, {pos:,} nucleosomal ({pos / n:.1%}), {n - pos:,} linker")
print(f"outer {K_FOLDS}-fold: ~{n // K_FOLDS:,} test sequences per fold; results -> {RUN.relative_to(ROOT)}")
gpu, procs = jobs.gpu_status()
print("GPU:", gpu)
print("GPU processes:", procs or "none")
print("MorphoNup jobs running:", jobs.running_jobs() or "none")

## Train

In [ ]:
if RUN_MODE == "background":
    LOG = jobs.launch(DATASET, K_FOLDS, SEEDS, allow_busy_gpu=ALLOW_BUSY_GPU)
    print("launched; log:", LOG.relative_to(ROOT))
elif RUN_MODE == "inline":
    gpu, procs = jobs.gpu_status()
    if procs and not ALLOW_BUSY_GPU:
        raise RuntimeError("GPU in use: " + "; ".join(procs))
    import tensorflow as tf
    from morphonup.cv import run
    from morphonup.data import Dataset
    ds = Dataset(config.dataset_folder(DATASET))
    for device in tf.config.list_physical_devices("GPU"):
        tf.config.set_logical_device_configuration(
            device, [tf.config.LogicalDeviceConfiguration(memory_limit=config.gpu_memory_mb(len(ds.y)))])
    run(ds, DATASET, K_FOLDS, SEEDS)
else:
    print("results_only: no training started")

## Progress

In [ ]:
display(jobs.progress(DATASET, K_FOLDS, SEEDS))
print(jobs.tail(jobs.log_path(DATASET, K_FOLDS, SEEDS)))

## Results

Pooled out-of-fold results over the outer test folds. **Refit** (trained on fit + tune rows) is the reported model; **base** (fit rows only, checkpoint chosen on tune rows) is shown for comparison.

In [ ]:
drawn = figures.ensure_figures(RUN, config.SETTINGS["epochs"]) if RUN.exists() else []
print(f"{len(drawn)} figures drawn now (missing or refreshed)")
summary = RUN / "summary" / "summary.md"
display(Markdown(summary.read_text()) if summary.exists() else Markdown("_No finished seed yet._"))

### Over seeds

In [ ]:
for name in ("seed_metrics", "roc_pr_seeds"):
    show(RUN / "summary" / "figures" / f"{name}.png")

### One seed (`SHOW_SEED`): reported model (refit)

In [ ]:
POOLED = RUN / f"seed_{SHOW_SEED}" / "pooled"
for name in ("roc_pr", "confusion_matrix", "reliability", "fold_metrics", "generalization_gap"):
    show(POOLED / "refit" / "figures" / f"{name}.png")

### One seed: base vs refit, and training

In [ ]:
for name in ("roc_pr_base_vs_refit", "fold_accuracy_base_vs_refit", "learning_curves_all_folds", "best_epochs"):
    show(POOLED / "figures" / f"{name}.png")

### One fold

In [ ]:
SHOW_FOLD = 1
FOLD = RUN / f"seed_{SHOW_SEED}" / f"fold_{SHOW_FOLD:02d}" / "figures"
for name in ("learning_curves", "confusion_matrix"):
    show(FOLD / f"{name}.png")

### Every figure of this run (PNG 600 dpi + PDF + CSV of the plotted values)

In [ ]:
cat = figures.catalog(RUN)
print(len(cat), "figures")
display(cat[["folder", "figure", "suggested_caption"]].head(60))